# Payment Guard - extractor 0.5B (LoRA, Unsloth)
Runtime > T4 GPU. El modelo solo EXTRAE campos; nunca decide pagos.

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

In [ ]:
import os, getpass
HF_USER = "P1p2gamer26"
MODEL_ID = f"{HF_USER}/payment-guard-extractor-0.5b"
BASE = "unsloth/Qwen2.5-0.5B-Instruct"  # si sobra tiempo: unsloth/Qwen2.5-1.5B-Instruct
# Cambia por el raw del repo publico una vez publicado (o sube ml/train.jsonl con el panel de archivos de Colab)
TRAIN_URL = "https://raw.githubusercontent.com/P1p2gamer26/payment-guard/main/ml/train.jsonl"
if not os.path.exists("train.jsonl"):
    !wget -q -O train.jsonl $TRAIN_URL || echo "descarga fallida: sube train.jsonl a mano"
!wc -l train.jsonl

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=2048, load_in_4bit=False)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"], random_state=30)
ds = load_dataset("json", data_files="train.jsonl", split="train")
ds = ds.map(lambda b: {"text": [tok.apply_chat_template(m, tokenize=False) for m in b["messages"]]}, batched=True)
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds, args=SFTConfig(
    dataset_text_field="text", max_seq_length=2048, per_device_train_batch_size=4, gradient_accumulation_steps=4,
    num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="linear", warmup_ratio=0.05, logging_steps=5,
    fp16=True, optim="adamw_8bit", output_dir="out", report_to="none", seed=30))
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
trainer.train()

In [ ]:
# prueba rapida con una factura nueva
FastLanguageModel.for_inference(model)
import json
from unsloth import get_chat_template
msgs = json.loads(open("train.jsonl").readline())["messages"][:2]
x = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
print(tok.decode(model.generate(input_ids=x, max_new_tokens=200, do_sample=False)[0][x.shape[1]:], skip_special_tokens=True))

In [ ]:
# Token HF con permiso write: se pide aqui, NO se guarda en el cuaderno.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = getpass.getpass("HF token (write): ")
model.push_to_hub_gguf(MODEL_ID, tok, quantization_method="q4_k_m", token=HF_TOKEN)
print("https://huggingface.co/" + MODEL_ID)